# Member 3: Behavior Policy Estimation, Off-Policy Evaluation (WIS), and Bootstrapping

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('..')
from member_3_evaluation.behavior_policy import BehaviorPolicyEstimator
from member_3_evaluation.off_policy_wis import WeightedImportanceSampler
from member_3_evaluation.bootstrap_evaluator import BootstrapEvaluator
from member_3_evaluation.sensitivity import SensitivityAnalyzer
from member_3_evaluation.visualizer import EvaluationVisualizer

os.makedirs('../figures', exist_ok=True)
os.makedirs('../data', exist_ok=True)


### 1. Load Trajectories and Trained AI Policy

In [ ]:
tr_train = pd.read_csv('../data/trajectories_train.csv')
tr_test = pd.read_csv('../data/trajectories_test.csv')

with open('../artifacts/ai_policy.json', 'r') as f:
    policy_data = json.load(f)
ai_policy = np.array(policy_data['policy'])

print("Train trajectories:", len(tr_train))
print("Test trajectories (held-out):", len(tr_test))
print("AI policy states loaded:", len(ai_policy))


### 2. Fit Clinician Behavior Policy with Laplace Smoothing

In [ ]:
behavior_estimator = BehaviorPolicyEstimator(n_states=75, n_actions=25, alpha=0.05)
behavior_estimator.fit(tr_train)

print("Behavior policy matrix shape:", behavior_estimator.probs.shape)
print("Minimum probability (smoothing floor):", float(np.min(behavior_estimator.probs)))


### 3. Trajectory-Wise Weighted Importance Sampling (WIS)

In [ ]:
wis_sampler = WeightedImportanceSampler(
    ai_policy=ai_policy,
    behavior_policy=behavior_estimator,
    gamma=0.99,
    soft_prob=0.99,
    weight_clip=500.0
)

eval_res = wis_sampler.evaluate_trajectories(tr_test)
print(f"Point Clinician Policy Value: {eval_res['clinician_value']:.2f}")
print(f"Point AI Policy Value (WIS):  {eval_res['wis_value']:.2f}")
print(f"Effective Sample Size (ESS):  {eval_res['ess']:.2f} / {eval_res['n_patients']}")


### 4. Patient-Level Bootstrap Resampling for 95% Confidence Intervals

In [ ]:
boot_evaluator = BootstrapEvaluator(wis_sampler, n_bootstraps=500, random_state=42)
boot_results = boot_evaluator.evaluate(tr_test)

print(f"Clinician Value: {boot_results['clin_median']:.2f} (95% CI: [{boot_results['clin_ci_lower']:.2f}, {boot_results['clin_ci_upper']:.2f}])")
print(f"AI Policy Value: {boot_results['wis_median']:.2f} (95% CI: [{boot_results['wis_ci_lower']:.2f}, {boot_results['wis_ci_upper']:.2f}])")


### 5. Sensitivity Analysis

In [ ]:
sens_analyzer = SensitivityAnalyzer(ai_policy, tr_train, tr_test, gamma=0.99)
sens_df = sens_analyzer.run_full_sensitivity_suite()
sens_df.to_csv('../data/sensitivity_analysis.csv', index=False)
sens_df


### 6. Generate Publication Figures

In [ ]:
vis = EvaluationVisualizer(output_dir='../figures')

f1 = vis.plot_policy_comparison(boot_results)
f2 = vis.plot_action_distributions(tr_test, ai_policy)
f3 = vis.plot_dose_difference_vs_mortality(tr_test, ai_policy)

print("Saved figures:")
print("-", f1)
print("-", f2)
print("-", f3)


### 7. Display Generated Figures

In [ ]:
from IPython.display import Image, display

display(Image(filename='../figures/policy_value_comparison.png'))
display(Image(filename='../figures/action_distribution.png'))
display(Image(filename='../figures/dose_excess_vs_mortality.png'))
